In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
import torch
import pandas as pd
from transformers import T5Tokenizer, T5EncoderModel
from Bio import SeqIO


# Load ProtT5
tokenizer = T5Tokenizer.from_pretrained(
    "Rostlab/prot_t5_xl_bfd",
    do_lower_case=False
)

model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_bfd",
    torch_dtype=torch.float16
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()


# Read FASTA
fasta_file = "/content/drive/MyDrive/Colab Notebooks/Dataset/training1_positive.fasta"

sequences = [
    (record.id, str(record.seq))
    for record in SeqIO.parse(fasta_file, "fasta")
]


def preprocess(sequence):
    allowed = "ACDEFGHIKLMNPQRSTVWY"
    sequence = sequence.upper()
    sequence = ''.join(
        aa if aa in allowed else "X"
        for aa in sequence
    )
    return " ".join(sequence)


features = []
ids = []


with torch.inference_mode():

    for seq_id, seq in sequences:

        processed_seq = preprocess(seq)

        inputs = tokenizer(
            processed_seq,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        embedding = outputs.last_hidden_state.squeeze(0)


        mask = attention_mask.squeeze(0).unsqueeze(-1)

        embedding = embedding * mask

        avg_embedding = (
            embedding.sum(dim=0) /
            mask.sum()
        )


        features.append(avg_embedding.cpu().numpy())
        ids.append(seq_id)


# Save embeddings

feature_df = pd.DataFrame(
    features,
    index=ids
)

feature_df.to_csv("ProtT5_Train1_DS1.csv")

print("ProtT5 feature extraction completed")
print(feature_df.shape)